# Breaking Hardware AES on CW305 FPGA (and other FPGA targets).

This tutorial relies on previous knowledge from the SCA101 course notebooks (in `../courses/sca101/`); make sure you go through these first to understand how a CPA attack works.

In this notebook, we'll apply knowledge from sca101 to break a hardware AES implementation on the CW305 Artix FPGA.

Some out-of-date background on the target FPGA project is can be found here: [Tutorial CW305-1 Building a Project](http://wiki.newae.com/Tutorial_CW305-1_Building_a_Project) (ignore the "capture setup" section, which uses the obsolete ChipWhisperer GUI; this notebook shows all you need to know about capture setup on the CW305 with Jupyter).

While this notebook has "CW305" in its title, it can also be used with the CW312T-A35 and CW312T-iCE40 targets.

## Background Theory
During this tutorial, we'll be working with a hardware AES implementation. This type of attack can be much more difficult than a software AES attack. In the software AES attacks, we needed hundreds or thousands of clock cycles to capture the algorithm's full execution. In contrast, a hardware AES implementation may have a variety of speeds. Depending on the performance of the hardware, a whole spectrum of execution speeds can be achieved by executing many operations in a single clock cycle. It is theoretically possible to execute the entire AES encryption in a single cycle, given enough hardware space and provided that the clock is not too fast. Most hardware accelerators are designed to complete one round or one large part of a round in a single cycle.

This fast execution may cause problems with a regular CPA attack. In software, we found that it was easy to search for the outputs of the s-boxes because these values would need to be loaded from memory onto a high-capacitance data bus. This is not necessarily true on an FPGA, where the output of the s-boxes may be directly fed into the next stage of the algorithm. In general, we may need some more knowledge of the hardware implementation to successfully complete an attack.

In our case, let's suppose that every round of AES is completed in a single clock cycle. Recall the execution of AES:

<img src="img/aes_operations.png" width="250">

Here, every blue block is executed in one clock cycle. This means that an excellent candidate for a CPA attack is the difference between the input and output of the final round. It is likely that this state is stored in a port that is updated every round, so we expect that the Hamming distance between the round input and output is the most important factor on the power consumption. Also, the last round is the easiest to attack because it has no MixColumns operation. We'll use this Hamming distance as the target in our CPA attack.

## Capture Notes

Most of the capture settings used below are similar to the standard ChipWhisperer scope settings. However, there are a couple of interesting points:

- We're only capturing 129 samples (the minimum allowed with CW-lite), and the encryption is completed in less than 60 samples with an x4 ADC clock. This makes sense - as we mentioned above, our AES implementation is computing each round in a single clock cycle.
- We're using EXTCLK x4 for our ADC clock. This means that the FPGA is outputting a clock signal, and we aren't driving it.

Other than these, the last interesting setting is the number of traces. By default, the capture software is ready to capture 5000 traces - many more than were required for software AES! It is difficult for us to measure the small power spikes from the Hamming distance on the last round: these signals are dwarfed by noise and the other operations on the chip. To deal with this small signal level, we need to capture many more traces.

## Capture Setup

Setup is somewhat similar to other targets, except that we are using an external clock (driven from the FPGA-- unless you're using the CW312T-A35 target). We'll also do the rest of the setup manually:

In [1]:
import chipwhisperer as cw
scope = cw.scope()
scope.default_setup()
if scope._is_husky:
    scope.adc.samples = 80
else:
    scope.adc.samples = 129
scope.adc.offset = 0
scope.adc.basic_mode = "rising_edge"
scope.trigger.triggers = "tio4"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2 = "disabled"

scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen                   
scope.io.cdc_settin

Next, we connect to the target board and program it with the correct bitfile.

If you have a CW305, you must correctly specify the variant (either 'CW305_35t' or 'CW305_100t').

In [2]:
TARGET_PLATFORM = 'CW305_100t'
#TARGET_PLATFORM = 'CW305_35t'

#TARGET_PLATFORM = 'CW312T_A35'
#TARGET_PLATFORM = 'CW312T_ICE40'

In [3]:
if TARGET_PLATFORM in ['CW312T_A35', 'CW312T_ICE40']:
    scope.io.hs2 = 'clkgen'
    fpga_id = None # not needed
    if TARGET_PLATFORM == 'CW312T_A35':
        platform = 'ss2_a35'
        scope.gain.db = 45 # this is a good setting for the inductive shunt; if using another, adjust as needed
    else:
        platform = 'ss2_ice40'
        scope.gain.db = 15
else:
    scope.gain.db = 25
    scope.io.hs2 = "disabled"
    platform = 'cw305'
    if TARGET_PLATFORM == 'CW305_100t':
        fpga_id = '100t'
    elif TARGET_PLATFORM == 'CW305_35t':
        fpga_id = '35t'

# On the CW305, setting force=False only programs the FPGA if it is currently unprogrammed, whereas force=True programs the FPGA regardless.
# This option isn't available on the CW312T_A35 or CW312T_ICE40.

#target = cw.target(scope, cw.targets.CW305, force=True, fpga_id=fpga_id, platform=platform)



path = r'C:/Users/91883/Documents/project_1_gf_masked_FINAL/project_1_gf_masked/project_1_lut_masked.runs/impl_1/cw305_top.bit'
target = cw.target(scope, cw.targets.CW305, force=True, fpga_id=fpga_id, platform=platform, bsfile=path)


(ChipWhisperer NAEUSB WARNING|File naeusb.py:826) Your firmware (0.51.0) is outdated - latest is 0.53.0 See https://chipwhisperer.readthedocs.io/en/latest/firmware.html for more information
(ChipWhisperer Target WARNING|File CW305.py:591) Using default Verilog defines (C:\Users\91883\ChipWhisperer\chipwhisperer\software\chipwhisperer/hardware/firmware/cw305/cw305_aes_defines.v); if this is not what you want, provide them via the defines_files argument


Next we set all the PLLs. We enable CW305's PLL1; this clock will feed both the target and the CW ADC. As explained [here](http://wiki.newae.com/Tutorial_CW305-1_Building_a_Project#Capture_Setup), **make sure the DIP switches on the CW305 board are set as follows**:
- J16 = 0
- K16 = 1

In [4]:
if TARGET_PLATFORM in ['CW305_100t', 'CW305_35t']:
    target.vccint_set(1.0)
    # we only need PLL1:
    target.pll.pll_enable_set(True)
    target.pll.pll_outenable_set(False, 0)
    target.pll.pll_outenable_set(True, 1)
    target.pll.pll_outenable_set(False, 2)

    # run at 10 MHz:
    target.pll.pll_outfreq_set(10E6, 1)

    # 1ms is plenty of idling time
    target.clkusbautooff = True
    target.clksleeptime = 1

CW-Husky requires a different setup when the ADC clock is driven by the target:

In [5]:
if TARGET_PLATFORM in ['CW305_100t', 'CW305_35t']:
    if scope._is_husky:
        scope.clock.clkgen_freq = 10e6
        scope.clock.clkgen_src = 'extclk'
        scope.clock.adc_mul = 4
        # if the target PLL frequency is changed, the above must also be changed accordingly
    else:
        scope.clock.adc_src = "extclk_x4"

If using the CW312T-A35 target, the capture hardware needs to drive the target clock:

In [6]:
if TARGET_PLATFORM in ['CW312T_A35', 'CW312T_ICE40']:
    scope.clock.clkgen_freq = 7.37e6
    scope.io.hs2 = 'clkgen'
    if scope._is_husky:
        scope.clock.clkgen_src = 'system'
        scope.clock.adc_mul = 4
        scope.clock.reset_dcms()
    else:
        scope.clock.adc_src = "clkgen_x4"
    import time
    time.sleep(0.1)
    target._ss2_test_echo()

Finally, ensure the ADC clock is locked:

In [7]:
import time
for i in range(5):
    scope.clock.reset_adc()
    time.sleep(1)
    if scope.clock.adc_locked:
        break 
assert (scope.clock.adc_locked), "ADC failed to lock"

Occasionally the ADC will fail to lock on the first try; when that happens, the above assertion will fail (and on the CW-Lite, the red LED will be on). Simply re-running the above cell again should fix things.

## # Step 2: Verify trigger/busy signal before capture


In [8]:
# Step 2: Verify trigger/busy signal before capture
import time
from Crypto.Cipher import AES

# do a single test encryption and check the trigger actually fires
ktp_test = cw.ktp.Basic()
test_key, test_text = ktp_test.next()

target.simpleserial_write('k', test_key)   # <-- send the key first!
scope.arm()
target.simpleserial_write('p', test_text)

ret = scope.capture()
trig_count = scope.adc.trig_count

print(f"scope.capture() timed out: {ret}")
print(f"Trigger sample count (trig_count): {trig_count}")

if trig_count == 0:
    print("FAIL: trig_count is 0 — trigger never fired. Check tio4 wiring / aes_busy signal.")
elif ret:
    print("WARNING: capture() reported a timeout — trigger may have fired but capture didn't complete cleanly.")
else:
    print("PASS: Trigger fired and capture completed normally.")

# read back ciphertext to confirm target is alive and responding
output = target.simpleserial_read('r', 16)
print(f"Plaintext sent : {list(test_text)}")
print(f"Ciphertext recv: {list(output)}")

cipher_check = AES.new(bytes(test_key), AES.MODE_ECB)
expected = list(cipher_check.encrypt(bytes(test_text)))
print(f"Expected CT    : {expected}")

if list(output) == expected:
    print("PASS: Ciphertext matches expected — target encrypting correctly.")
else:
    print("FAIL: Ciphertext mismatch — target is NOT encrypting correctly.")

scope.capture() timed out: False
Trigger sample count (trig_count): 10324
PASS: Trigger fired and capture completed normally.
Plaintext sent : [26, 137, 241, 72, 90, 58, 93, 0, 143, 126, 91, 183, 150, 143, 154, 147]
Ciphertext recv: [143, 138, 254, 9, 118, 228, 144, 131, 115, 179, 160, 100, 194, 151, 199, 146]
Expected CT    : [143, 138, 254, 9, 118, 228, 144, 131, 115, 179, 160, 100, 194, 151, 199, 146]
PASS: Ciphertext matches expected — target encrypting correctly.


## Trace Capture
Below is the capture loop. The main body of the loop loads some new plaintext, arms the scope, sends the key and plaintext, then finally records and appends our new trace to the `traces[]` list.

Because we're capturing 5000 traces, this takes a bit longer than the attacks against software AES implementations.

Note that the encryption result is read from the target and compared to the expected results, as a sanity check.

In [9]:
project_file = "projects/Tutorial_HW_CW305.cwp"
project = cw.create_project(project_file, overwrite=True)

In [10]:
from tqdm.notebook import tnrange
import numpy as np
import time
from Crypto.Cipher import AES

ktp = cw.ktp.Basic()

traces = []
textin = []
keys = []
N = 10C000  # Number of traces

# initialize cipher to verify DUT result:
key, text = ktp.next()
cipher = AES.new(bytes(key), AES.MODE_ECB)

for i in tnrange(N, desc='Capturing traces'):
    # run aux stuff that should come before trace here

    key, text = ktp.next()  # manual creation of a key, text pair can be substituted here
    textin.append(text)
    keys.append(key)
    
    ret = cw.capture_trace(scope, target, text, key)
    if not ret:
        print("Failed capture")
        continue

    assert (list(ret.textout) == list(cipher.encrypt(bytes(text)))), "Incorrect encryption result!\nGot {}\nExp {}\n".format(ret.textout, list(text))
    #trace += scope.getLastTrace()
        
    traces.append(ret.wave)
    project.traces.append(ret)

Capturing traces:   0%|          | 0/2000 [00:00<?, ?it/s]

This shows how a captured trace can be plotted:

In [11]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook

output_notebook()
p = figure(width=800)

xrange = list(range(len(traces[0])))
p.line(xrange, traces[0], line_color="red")
show(p)

Loading BokehJS ...

Finally we save our traces and disconnect. By saving the traces, the attack can be repeated in the future without having to repeat the trace acquisition steps above.

## step3 Print power value per trace (LUT / GF - same cell works for both)

In [12]:
import numpy as np
import pandas as pd

traces_np = np.array(traces)

# Save all traces to Excel
df = pd.DataFrame(traces_np)

df.to_excel("all_traces_lut_maked.xlsx", index=False)

print(f"Total traces captured: {len(traces_np)}")
print(f"Samples per trace: {traces_np.shape[1]}")
print("All traces saved to all_traces.xlsx")

Total traces captured: 2000
Samples per trace: 80
All traces saved to all_traces.xlsx


##  Step 4a: TVLA Capture - fixed vs random plaintext (interleaved)

In [13]:

from tqdm.notebook import tnrange
import numpy as np
from Crypto.Cipher import AES

N_TVLA = 10000  # total traces, half fixed half random (interleaved)

# fixed plaintext (constant across all "fixed" captures)
fixed_text = bytearray([0x00, 0x11, 0x22, 0x33, 0x44, 0x55, 0x66, 0x77,
                         0x88, 0x99, 0xAA, 0xBB, 0xCC, 0xDD, 0xEE, 0xFF])

# use the same key for the whole TVLA run (key must stay fixed for TVLA)
tvla_key, _ = ktp.next()
cipher_tvla = AES.new(bytes(tvla_key), AES.MODE_ECB)

tvla_traces_fixed = []
tvla_traces_random = []
tvla_labels = []  # 1 = fixed, 0 = random, in capture order (for reference)

ktp_random = cw.ktp.Basic()

for i in tnrange(N_TVLA, desc='TVLA Capturing'):
    if i % 2 == 0:
        # fixed plaintext trace
        text = fixed_text
        is_fixed = 1
    else:
        # random plaintext trace
        _, text = ktp_random.next()
        is_fixed = 0

    ret = cw.capture_trace(scope, target, text, tvla_key)
    if not ret:
        print(f"Failed capture at trace {i}")
        continue

    assert (list(ret.textout) == list(cipher_tvla.encrypt(bytes(text)))), \
        "Incorrect encryption result!\nGot {}\nExp {}\n".format(ret.textout, list(text))

    if is_fixed:
        tvla_traces_fixed.append(ret.wave)
    else:
        tvla_traces_random.append(ret.wave)
    tvla_labels.append(is_fixed)

tvla_traces_fixed = np.array(tvla_traces_fixed)
tvla_traces_random = np.array(tvla_traces_random)

print(f"Fixed set: {tvla_traces_fixed.shape[0]} traces")
print(f"Random set: {tvla_traces_random.shape[0]} traces")

TVLA Capturing:   0%|          | 0/2000 [00:00<?, ?it/s]

Fixed set: 1000 traces
Random set: 1000 traces


## Step 4b: TVLA - Welch t-test, 1st order

In [14]:
# Step 4b: TVLA - Welch t-test, 1st order
import numpy as np
from scipy import stats

# 1st order: t-test directly on raw sample points, per time sample
t_stat_1st, p_val_1st = stats.ttest_ind(tvla_traces_fixed, tvla_traces_random,
                                         axis=0, equal_var=False)

print(f"1st order t-stat shape: {t_stat_1st.shape}")
print(f"Max |t-stat|: {np.max(np.abs(t_stat_1st)):.4f}  (threshold for leakage is |t| > 4.5)")

n_leaking_points = np.sum(np.abs(t_stat_1st) > 4.5)
print(f"Number of sample points exceeding |t|=4.5: {n_leaking_points} / {len(t_stat_1st)}")

from bokeh.plotting import figure, show
from bokeh.io import output_notebook
output_notebook()

p = figure(width=900, height=400, title="1st Order TVLA - Welch t-test")
xrange = list(range(len(t_stat_1st)))
p.line(xrange, t_stat_1st, line_color="blue", legend_label="t-statistic")
p.line(xrange, [4.5]*len(xrange), line_color="red", line_dash="dashed", legend_label="+4.5 threshold")
p.line(xrange, [-4.5]*len(xrange), line_color="red", line_dash="dashed", legend_label="-4.5 threshold")
show(p)

1st order t-stat shape: (80,)
Max |t-stat|: nan  (threshold for leakage is |t| > 4.5)
Number of sample points exceeding |t|=4.5: 0 / 80


C:\Users\91883\ChipWhisperer\cwenv\Lib\site-packages\scipy\stats\_axis_nan_policy.py:618: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, axis=axis, **kwds)


Loading BokehJS ...

## Step 4c: TVLA - Welch t-test, 2nd order

In [15]:
# Step 4c: TVLA - Welch t-test, 2nd order
import numpy as np
from scipy import stats

# 2nd order: mean-center each set, square it, then t-test on that
fixed_centered = tvla_traces_fixed - np.mean(tvla_traces_fixed, axis=0)
random_centered = tvla_traces_random - np.mean(tvla_traces_random, axis=0)

fixed_squared = fixed_centered ** 2
random_squared = random_centered ** 2

t_stat_2nd, p_val_2nd = stats.ttest_ind(fixed_squared, random_squared,
                                         axis=0, equal_var=False)

print(f"2nd order t-stat shape: {t_stat_2nd.shape}")
print(f"Max |t-stat|: {np.max(np.abs(t_stat_2nd)):.4f}  (threshold for leakage is |t| > 4.5)")

n_leaking_points_2nd = np.sum(np.abs(t_stat_2nd) > 4.5)
print(f"Number of sample points exceeding |t|=4.5: {n_leaking_points_2nd} / {len(t_stat_2nd)}")

p2 = figure(width=900, height=400, title="2nd Order TVLA - Welch t-test")
xrange2 = list(range(len(t_stat_2nd)))
p2.line(xrange2, t_stat_2nd, line_color="green", legend_label="t-statistic")
p2.line(xrange2, [4.5]*len(xrange2), line_color="red", line_dash="dashed", legend_label="+4.5 threshold")
p2.line(xrange2, [-4.5]*len(xrange2), line_color="red", line_dash="dashed", legend_label="-4.5 threshold")
show(p2)

2nd order t-stat shape: (80,)
Max |t-stat|: nan  (threshold for leakage is |t| > 4.5)
Number of sample points exceeding |t|=4.5: 0 / 80


In [18]:
print(np.argmax(np.abs(t_stat_1st)), np.max(np.abs(t_stat_1st)))

8 nan


In [19]:
project.save()
scope.dis()
target.dis()

(ChipWhisperer Scope ERROR|File naeusbchip.py:113) Scope already disconnected!


## Attack
Now we re-open our saved project and specify the attack parameters. For this hardware AES implementation, we use a different leakage model and attack than what is used for the software AES implementations.

Note that this attack requires only the ciphertext, not the plaintext.

In [20]:
import chipwhisperer as cw
import chipwhisperer.analyzer as cwa
project_file = "projects/Tutorial_HW_CW305"
project = cw.open_project(project_file)
attack = cwa.cpa(project, cwa.leakage_models.last_round_state_diff)
cb = cwa.get_jupyter_callback(attack)

This runs the attack:

In [21]:
attack_results = attack.run(cb)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
PGE=,157,131,104,41,29,179,179,109,131,173,212,7,108,225,90,112
0,180.086,CB0.094,190.100,5E0.096,AC0.097,2E0.090,550.097,D10.091,EF0.090,DC0.087,580.089,F10.097,8F0.088,590.104,A10.092,B50.097
1,980.084,F50.088,FD0.092,EB0.090,800.095,F90.085,7E0.094,4A0.087,2B0.087,890.082,720.086,0A0.091,370.088,2F0.081,290.090,F30.086
2,630.080,B80.086,3F0.089,360.085,5C0.092,810.085,620.085,700.085,A00.086,020.081,C40.085,590.089,C20.086,360.079,A00.089,FD0.085
3,050.080,E10.084,DA0.089,190.083,220.091,210.084,A10.081,0A0.081,1F0.080,6A0.080,B90.083,7E0.089,870.085,AA0.078,3E0.084,D50.083
4,B80.078,870.083,BA0.087,AD0.083,9A0.088,820.081,240.080,180.079,8B0.080,CC0.079,C00.081,460.086,9B0.080,120.077,0B0.083,270.083


The attack results can be saved for later viewing or processing without having to repeat the attack:

In [ ]:
import pickle
pickle_file = project_file + ".results.pickle"
pickle.dump(attack_results, open(pickle_file, "wb"))

You may notice that we didn't get the expected key from this attack, but still got a good difference in correlation between the best guess and the next best guess. This is because we actually recovered the key from the last round of AES. We'll need to use analyzer to get the actual AES key: 

In [ ]:
from chipwhisperer.analyzer.attacks.models.aes.key_schedule import key_schedule_rounds
recv_lastroundkey = [kguess[0][0] for kguess in attack_results.find_maximums()]
recv_key = key_schedule_rounds(recv_lastroundkey, 10, 0)
for subkey in recv_key:
    print(hex(subkey))

## Tests
Check that the key obtained by the attack is the key that was used.
This attack targets the last round key, so we have to roll it back to compare against the key we provided.

In [ ]:
key = list(key)
assert (key == recv_key), "Failed to recover encryption key\nGot:      {}\nExpected: {}".format(recv_key, key)

In [ ]:
import pandas as pd
import numpy as np
from Crypto.Cipher import AES

rows = []

for i in range(len(traces)):
    row = {}

    # Trace Number
    row["Trace#"] = i

    # Plaintext Bytes
    for j in range(16):
        row[f"PT[{j}]"] = textin[i][j]

    # Key Bytes
    for j in range(16):
        row[f"K[{j}]"] = keys[i][j]

    # Ciphertext Bytes
    cipher = AES.new(bytes(keys[i]), AES.MODE_ECB)
    ct = cipher.encrypt(bytes(textin[i]))

    for j in range(16):
        row[f"CT[{j}]"] = ct[j]

    # Power Trace Samples
    for j, sample in enumerate(traces[i]):
        row[f"S{j}"] = float(sample)

    rows.append(row)

df = pd.DataFrame(rows)

df.to_excel("AES_HW_CW305_Full_Dataset_LUT.xlsx", index=False)

print(f"Saved {len(traces)} traces to AES_HW_CW305_Full_Dataset_lut.xlsx")
print(f"Samples per trace: {len(traces[0])}")

## Next steps

The `jupyter/demos/CW305_ECC/` folder contains a series of tutorials for attacking hardware ECC on the CW305.

This CW305 appnote contains additional details on the CW305 platform: http://media.newae.com/appnotes/NAE0010_Whitepaper_CW305_AES_SCA_Attack.pdf